# Week 4 – Day 5: Dashboard and Final Validation

This notebook performs final end-to-end validation of the GeoValuation AI application.

Validation includes:

- Final dashboard artifacts
- Trained XGBoost model
- Model comparison results
- Property-level prediction outputs
- Feature consistency
- Prediction validity
- Dashboard source structure
- Final deployment readiness

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

print("=" * 70)
print("WEEK 4 DAY 5 – FINAL DASHBOARD VALIDATION")
print("=" * 70)

BASE_DIR = Path("..")

DATA_DIR = (BASE_DIR / "data" / "processed")

WEEK3_DIR = (DATA_DIR / "week-3")

GNN_DIR = (WEEK3_DIR / "gnn")

MODEL_DIR = (BASE_DIR / "models")

APP_DIR = (BASE_DIR / "app")

print("Project root:", BASE_DIR.resolve())

WEEK 4 DAY 5 – FINAL DASHBOARD VALIDATION
Project root: E:\AARAV\Infotact-DS-ML\Project-3-Geospatial-Valuation


In [2]:
NODES_FILE = (WEEK3_DIR / "final_graph_nodes.csv")

TARGETS_FILE = (WEEK3_DIR / "final_graph_targets.csv")

MODEL_FILE = (MODEL_DIR / "xgboost_comparison.pkl")

COMPARISON_FILE = (GNN_DIR / "final_model_comparison.csv")

ERROR_FILE = (GNN_DIR / "final_prediction_error_analysis.csv")

PRICE_RANGE_FILE = (GNN_DIR / "final_price_range_analysis.csv")

DASHBOARD_FILE = (GNN_DIR / "dashboard_predictions.csv")

required_files = [
    NODES_FILE,
    TARGETS_FILE,
    MODEL_FILE,
    COMPARISON_FILE,
    ERROR_FILE,
    PRICE_RANGE_FILE,
    DASHBOARD_FILE,
]

print("\nRequired artifacts:")

for file_path in required_files:

    print(
        f"{file_path.name:<45} "
        f"{'FOUND ✅' if file_path.exists() else 'MISSING ❌'}"
    )

assert all(file_path.exists() for file_path in required_files)

print("\n✅ All final artifacts are present.")


Required artifacts:
final_graph_nodes.csv                         FOUND ✅
final_graph_targets.csv                       FOUND ✅
xgboost_comparison.pkl                        FOUND ✅
final_model_comparison.csv                    FOUND ✅
final_prediction_error_analysis.csv           FOUND ✅
final_price_range_analysis.csv                FOUND ✅
dashboard_predictions.csv                     FOUND ✅

✅ All final artifacts are present.


In [3]:
nodes_df = pd.read_csv(NODES_FILE)

targets_df = pd.read_csv(TARGETS_FILE)

model_comparison = pd.read_csv(COMPARISON_FILE)

error_analysis = pd.read_csv(ERROR_FILE)

price_range_analysis = pd.read_csv(PRICE_RANGE_FILE)

dashboard_predictions = pd.read_csv(DASHBOARD_FILE)

model = joblib.load(MODEL_FILE)

print("Nodes shape:", nodes_df.shape)
print("Targets shape:", targets_df.shape)
print("Dashboard predictions shape:",dashboard_predictions.shape)
print("Model type:",type(model).__name__)

Nodes shape: (21613, 26)
Targets shape: (21613, 2)
Dashboard predictions shape: (3242, 10)
Model type: XGBRegressor


In [4]:
feature_columns = [col for col in nodes_df.columns if col not in ["node_id","id",]]

print("Number of model features:",len(feature_columns))

print("\nFeature columns:")
for i, feature in enumerate(feature_columns,start=1):
    print(f"{i:02d}. {feature}")

assert len(feature_columns) == 24
assert len(nodes_df) == 21613
assert len(targets_df) == 21613

print("\n✅ Feature count and dataset dimensions validated.")

Number of model features: 24

Feature columns:
01. bedrooms
02. bathrooms
03. sqft_living
04. sqft_lot
05. floors
06. waterfront
07. view
08. condition
09. grade
10. sqft_above
11. sqft_basement
12. House Age
13. yr_built
14. yr_renovated
15. lat
16. long
17. lat_scaled
18. long_scaled
19. mean_neighbor_distance_scaled
20. median_neighbor_distance_scaled
21. min_neighbor_distance_scaled
22. max_neighbor_distance_scaled
23. std_neighbor_distance_scaled
24. neighbor_count

✅ Feature count and dataset dimensions validated.


In [5]:
assert nodes_df["node_id"].is_unique
assert targets_df["node_id"].is_unique

assert set(nodes_df["node_id"]) == set(targets_df["node_id"])

print("✅ Node and target alignment validated.")

✅ Node and target alignment validated.


In [6]:
X_full = nodes_df[feature_columns].copy()

y_full = targets_df["price"].copy()

assert not X_full.isnull().any().any()
assert not y_full.isnull().any()

assert np.isfinite(X_full.to_numpy()).all()

assert np.isfinite(y_full.to_numpy()).all()

print(
    "✅ Feature matrix and target values contain "
    "no missing or infinite values."
)

print("\nFeature matrix shape:",X_full.shape)

print("Target shape:",y_full.shape)

✅ Feature matrix and target values contain no missing or infinite values.

Feature matrix shape: (21613, 24)
Target shape: (21613,)


In [7]:
full_predictions = model.predict(X_full)

assert len(full_predictions) == len(X_full)

assert np.isfinite(full_predictions).all()

print("✅ Full-dataset XGBoost prediction pipeline validated.")

print(f"Prediction minimum: ${full_predictions.min():,.2f}")

print(f"Prediction maximum: ${full_predictions.max():,.2f}")

print(f"Prediction mean:    ${full_predictions.mean():,.2f}")

✅ Full-dataset XGBoost prediction pipeline validated.
Prediction minimum: $92,095.71
Prediction maximum: $1,217,643.75
Prediction mean:    $511,453.53


In [8]:
dashboard_required_columns = [
    "node_id",
    "actual_price",
    "xgboost_prediction",
    "baseline_gnn_prediction",
    "attention_gnn_prediction",
    "selected_model",
    "selected_prediction",
    "prediction_error",
    "absolute_error",
    "percentage_error",
]

missing_dashboard_columns = [col for col in dashboard_required_columns if col not in dashboard_predictions.columns]

assert not missing_dashboard_columns, ("Missing dashboard columns: " + str(missing_dashboard_columns))

print("✅ Dashboard prediction schema validated.")

✅ Dashboard prediction schema validated.


In [9]:
assert dashboard_predictions["selected_prediction"].notna().all()

assert np.isfinite(dashboard_predictions["selected_prediction"]).all()

assert (dashboard_predictions["actual_price"] > 0).all()

assert (dashboard_predictions["percentage_error"] >= 0).all()

print("✅ Dashboard prediction values validated.")

✅ Dashboard prediction values validated.


In [10]:
best_model = (model_comparison.sort_values("MAPE").iloc[0])

print("=" * 70)
print("FINAL MODEL PERFORMANCE")
print("=" * 70)
print(f"Best model : {best_model['Model']}")
print(f"Test RMSE  : ${best_model['RMSE']:,.2f}")
print(f"Test MAPE  : {best_model['MAPE']:.2f}%")
print(f"Test MAE   : ${best_model['MAE']:,.2f}")
print(f"Test R²    : {best_model['R2']:.4f}")

assert best_model["Model"] == "XGBoost"

print("\n✅ XGBoost confirmed as current deployment candidate.")

FINAL MODEL PERFORMANCE
Best model : XGBoost
Test RMSE  : $75,232.36
Test MAPE  : 11.39%
Test MAE   : $51,576.17
Test R²    : 0.9038

✅ XGBoost confirmed as current deployment candidate.


In [11]:
print("=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

comparison_display = (model_comparison.sort_values("MAPE").reset_index(drop=True))

display(comparison_display)

MODEL COMPARISON


,Model,RMSE,MAPE,MAE,R2,RMSE_Change_vs_XGBoost_%,MAPE_Change_vs_XGBoost_%
0,XGBoost,75232.358681,11.387990,51576.165383,0.903846,0.000000,0.000000
1,Baseline GNN,80285.807116,12.317674,55964.859485,0.890494,6.717121,8.163725
2,Attention GNN,121562.172602,19.396719,87503.563609,0.748952,61.582296,70.326106


In [12]:
app_required_files = [
    APP_DIR / "app.py",
    APP_DIR / "config.py",
    APP_DIR / "assets" / "style.css",
    APP_DIR / "modules" / "__init__.py",
    APP_DIR / "utils" / "__init__.py",
    APP_DIR / "utils" / "data_loader.py",
    APP_DIR / "utils" / "prediction.py",
    APP_DIR / "utils" / "spatial.py",
    APP_DIR / "utils" / "ui.py",
    APP_DIR / "modules" / "overview.py",
    APP_DIR / "modules" / "prediction.py",
    APP_DIR / "modules" / "property_map.py",
    APP_DIR / "modules" / "neighborhood.py",
    APP_DIR / "modules" / "spatial_disparity.py",
    APP_DIR / "modules" / "model_comparison.py",
    APP_DIR / "modules" / "error_analysis.py",
    APP_DIR / "modules" / "what_if.py",
    APP_DIR / "modules" / "model_explanation.py",
]

print("Dashboard source validation:")

for file_path in app_required_files:
    relative_path = str(file_path.relative_to(BASE_DIR))
    status = ("FOUND ✅" if file_path.exists() else "MISSING ❌")
    print(f"{relative_path:<60}{status}")

assert all(file_path.exists() for file_path in app_required_files)

print("\n✅ Dashboard modular structure validated.")

Dashboard source validation:
app\app.py                                                  FOUND ✅
app\config.py                                               FOUND ✅
app\assets\style.css                                        FOUND ✅
app\modules\__init__.py                                     FOUND ✅
app\utils\__init__.py                                       FOUND ✅
app\utils\data_loader.py                                    FOUND ✅
app\utils\prediction.py                                     FOUND ✅
app\utils\spatial.py                                        FOUND ✅
app\utils\ui.py                                             FOUND ✅
app\modules\overview.py                                     FOUND ✅
app\modules\prediction.py                                   FOUND ✅
app\modules\property_map.py                                 FOUND ✅
app\modules\neighborhood.py                                 FOUND ✅
app\modules\spatial_disparity.py                            FOUND ✅
app\modules\model_c

In [13]:
dashboard_predictions["selected_model"].value_counts()

selected_model
XGBoost    3242
Name: count, dtype: int64

In [14]:
selected_model_counts = (dashboard_predictions["selected_model"].value_counts())

print("Selected model distribution:")

display(selected_model_counts.to_frame("Count"))

Selected model distribution:


,Count
selected_model,
XGBoost,3242


In [15]:
print("=" * 70)
print("FINAL DATASET SUMMARY")
print("=" * 70)

print(f"Properties available      : {len(nodes_df):,}")

print(f"Model features            : {len(feature_columns)}")

print(f"Dashboard prediction rows : " f"{len(dashboard_predictions):,}")

print(f"Best model                : " f"{best_model['Model']}")

print(f"Best test MAPE            : " f"{best_model['MAPE']:.2f}%")

print("\n✅ WEEK 4 DAY 5 VALIDATION PASSED")

FINAL DATASET SUMMARY
Properties available      : 21,613
Model features            : 24
Dashboard prediction rows : 3,242
Best model                : XGBoost
Best test MAPE            : 11.39%

✅ WEEK 4 DAY 5 VALIDATION PASSED


## Final Validation Outcome

The final dashboard artifacts, trained XGBoost model, feature matrix, prediction outputs, model comparison results, and modular Streamlit application structure were successfully validated.

The current deployment candidate is XGBoost based on the lowest controlled test MAPE.

Final controlled comparison:

- XGBoost: 11.39% MAPE
- Baseline GNN: 12.32% MAPE
- Attention GNN: 19.40% MAPE

The project is ready for final documentation and GitHub submission.